Auteur : Laetitia Ikusawa

Date : Mars 2026

Objectif : Update notebook P8_Notebook_Linux_EMR_PySpark_V1.0.ipynb to add broadcast Tensorflow and incude PCA
***

<details>
  <summary>Summary</summary>

  [Configuration](#configuration)
  
</details>



## Configuration


### Dependencies import

In [ ]:
# ============================================================
# LOGGING AND WARNINGS
# ============================================================

import os
import warnings
import logging

# No tensorflow warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # 0=all, 1=info, 2=warning, 3=error only

# No warnings
warnings.filterwarnings("ignore")

# Set logging level for PySpark/Py4J
logging.getLogger("py4j").setLevel(logging.ERROR)
logging.getLogger("pyspark").setLevel(logging.ERROR)


In [2]:
import sys

print(f"Python version: {sys.version}")

try:
    import pyspark

    print(f"PySpark version: {pyspark.__version__}")
except ImportError:
    print("⚠️ PySpark not installed")

try:
    import tensorflow as tf

    print(f"TensorFlow version: {tf.__version__}")
except ImportError:
    print("⚠️ TensorFlow not installed")

try:
    from PIL import Image

    print(f"PIL/Pillow: OK")
except ImportError:
    print("⚠️ Pillow not installed")

Python version: 3.11.15 (main, Mar  3 2026, 20:25:43) [GCC 14.2.0]
PySpark version: 3.5.3
TensorFlow version: 2.21.0
PIL/Pillow: OK


In [1]:
# PySpark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, element_at, split
from pyspark.sql.types import ArrayType, FloatType, StructType, StructField, StringType
from pyspark.ml.feature import PCA, VectorAssembler
from pyspark.ml.linalg import Vectors, VectorUDT

# TensorFlow imports
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras import Model

# Autres imports
from PIL import Image
import pandas as pd
import numpy as np
import io
import os
from pathlib import Path

print("✅ Imports réussis")

✅ Imports réussis


### Path definitions

In [2]:
# Automaticaly find project root
def find_project_root():
    """
    Find project root by searching for markers (.git, poetry.lock).
    Works regardless of where the Jupyter kernel is started.
    """
    # Try using __file__ if available (scripts .py)
    current = Path(__file__).parent if "__file__" in globals() else Path.cwd()

    # Search for root (contains .git or poetry.lock)
    for parent in [current] + list(current.parents):
        if (parent / ".git").exists() or (parent / "poetry.lock").exists():
            return parent

    # Fallback: if we are in notebooks/, go up one level
    if current.name == "notebooks":
        return current.parent
    elif (current / "notebooks").exists():
        return current

    # Last resort
    return current.parent


# Project paths
PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
FEATURES_DIR = DATA_DIR / "features"
PCA_DIR = DATA_DIR / "pca"

# Create directories if they don't exist
for directory in [DATA_DIR, RAW_DATA_DIR, FEATURES_DIR, PCA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"📁 Project directory: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"📁 Raw data directory: {RAW_DATA_DIR}")
print(f"📁 Features directory: {FEATURES_DIR}")
print(f"📁 PCA directory: {PCA_DIR}")

📁 Project directory: /app
📁 Data directory: /app/data
📁 Raw data directory: /app/data/raw
📁 Features directory: /app/data/features
📁 PCA directory: /app/data/pca


## Initialize PySpark

### Create SparkSession

In [14]:
spark = (
    SparkSession.builder.appName("P11-Fruits-Local-Development") # Name of the application that will be displayed in the Spark UI
    .master("local[*]") # Local development (all cores)
    .config("spark.driver.memory", "4g") # Driver memory (4GB)
    .config("spark.executor.memory", "4g") # Executor memory (4GB)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") # Arrow enabled (faster data processing)
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1024") # Max records per batch (1024)
    .getOrCreate()
)

# Set log level to WARN
spark.sparkContext.setLogLevel("WARN")

# Get SparkContext for broadcast
sc = spark.sparkContext

print(f"✅ SparkSession created")
print(f"   Version Spark: {spark.version}")
print(f"   Master: {spark.sparkContext.master}")
print(f"   App Name: {spark.sparkContext.appName}")

✅ SparkSession created
   Version Spark: 3.5.3
   Master: local[*]
   App Name: P11-Fruits-Local-Development


## Data Exploration

### Download data (optional)

In [ ]:
import json
from dotenv import load_dotenv

env_path = PROJECT_ROOT / ".env"
load_dotenv(dotenv_path=env_path)

kaggle_username = os.getenv("KAGGLE_USERNAME")
kaggle_key = os.getenv("KAGGLE_TOKEN")

assert kaggle_username and kaggle_key, "Variables KAGGLE_USERNAME / KAGGLE_KEY manquantes."

kaggle_dir = os.environ.get("KAGGLE_CONFIG_DIR", "/app/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)

print("kaggle.json created in", kaggle_json_path)

!kaggle datasets download -d moltean/fruits -p /app/data/raw --unzip

kaggle.json created in /app/.kaggle/kaggle.json
Dataset URL: https://www.kaggle.com/datasets/moltean/fruits
License(s): CC-BY-SA-4.0
100%|██████████████████████████████████████| 5.40G/5.40G [02:07<00:00, 45.4MB/s]



### Explore data


We uploaded the fruit directory into data/raw. We will use fruits-360-original-size for our production service. While developping, we will use fruits-360-100x100

In [20]:
# Get the number of classes in the training set
training_dir = DATA_DIR / "raw" / "fruits" / "fruits-360_dataset" / "fruits-360" / "Training"
test_dir = DATA_DIR / "raw" / "fruits" / "fruits-360_dataset" / "fruits-360" / "Test"

classes = [fruit_class for fruit_class in os.listdir(training_dir) if os.path.isdir(os.path.join(training_dir, fruit_class))]
print(f"Number of training classes: {len(classes)}")
print(f"Example of classes: {classes[:10]}")


# Get the number of images in the training set
num_train_images = sum(len(files) for r, d, files in os.walk(training_dir) if files)
print(f"Number of training images: {num_train_images}")

num_test_images = sum(len(files) for r, d, files in os.walk(test_dir) if files)
print(f"Number of test images: {num_test_images}")

total_images = num_train_images + num_test_images
print(f"Total number of images: {total_images}")


Number of training classes: 131
Example of classes: ['Tomato 4', 'Apple Red Delicious', 'Tomato 3', 'Huckleberry', 'Blueberry', 'Pear Red', 'Banana Lady Finger', 'Melon Piel de Sapo', 'Pear', 'Cherry 1']
Number of training images: 67692
Number of test images: 22688
Total number of images: 90380


### Load dataset with PySpark


In [34]:
# Like the previous notebook, load 300 images
MAX_IMAGES = 300

images = (
        spark.read.format("binaryFile")
        .option("pathGlobFilter", "*.jpg")
        .option("recursiveFileLookup", "true")
        .load(os.path.join(training_dir))
        .limit(MAX_IMAGES)
    )
print(f"Number of images loaded: {images.count()}")


print("Schema of the images DataFrame:")
images.printSchema()

# Let's display the first 5 images
images.show(5)


Number of images loaded: 300
Schema of the images DataFrame:
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)

+--------------------+--------------------+------+--------------------+
|                path|    modificationTime|length|             content|
+--------------------+--------------------+------+--------------------+
|file:/app/data/ra...|2026-03-13 14:07:...|  7437|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7434|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7424|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7423|[FF D8 FF E0 00 1...|
|file:/app/data/ra...|2026-03-13 14:07:...|  7416|[FF D8 FF E0 00 1...|
+--------------------+--------------------+------+--------------------+
only showing top 5 rows



In [35]:
# Add label column
images = images.withColumn("label", element_at(split(images["path"], "/"), -2))

# Keep Path and label columns
images = images.select("path", "label")

# Display the first 5 images
images.show(5)

# Display the schema of the images DataFrame
images.printSchema()

# Count the number of images per class
print(f"Classes distribution:")
label_counts = images.groupBy("label").count().orderBy("label")
label_counts.show(20, truncate=False)


+--------------------+--------------+
|                path|         label|
+--------------------+--------------+
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|Pineapple Mini|
|file:/app/data/ra...|     Raspberry|
|file:/app/data/ra...|     Raspberry|
+--------------------+--------------+
only showing top 5 rows

root
 |-- path: string (nullable = true)
 |-- label: string (nullable = true)

Classes distribution:


+--------------+-----+
|label         |count|
+--------------+-----+
|Pineapple Mini|143  |
|Raspberry     |157  |
+--------------+-----+

